# Multi-Strategy Regime-Based Approach

This notebook implements a regime-based Bitcoin accumulation strategy and compares it against a Uniform DCA baseline.

The overall workflow is:

1. Load the dataset.
2. Engineer rolling lookback features such as SMA ratios and BTC returns.
3. Classify each day into a combined market regime using BTC trend, MVRV valuation, and market-cap versus realized-cap growth.
4. Generate candidate strategy weights from StackSats MVRV, StackSats Momentum, and a custom SMA strategy.
5. Use training data only to learn the best candidate strategy for each regime.
6. Run a train-only grid search to select the best lookback and fallback strategy combination.
7. Evaluate the selected strategy on the test period only after final selection.
8. Build StackSats-compatible strategy objects for final train/test reporting.
9. Compare performance against Uniform DCA using sats-per-dollar, improvement percentage, win rate, and StackSats comparison outputs.
10. Plot the full train and test period using shared project plotting utilities.

The key goal is to test whether a learned regime-to-strategy mapping can improve Bitcoin accumulation relative to Uniform DCA.


In [1]:
# Author : Raghav Gupta
# ============================================================
# Cell 1: Imports, project-root setup, data preparation check,
# and strategy configuration
# ============================================================

import sys
from pathlib import Path
from itertools import product

import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# Project-root setup
# ============================================================

project_root = Path.cwd().resolve()

for parent in [project_root] + list(project_root.parents):
    if (parent / "src").exists():
        project_root = parent
        break
else:
    raise FileNotFoundError(
        "Could not find project root containing src folder. "
        f"Current working directory: {Path.cwd()}"
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Current working directory:", Path.cwd())
print("Project root:", project_root)
print("src exists:", (project_root / "src").exists())


# ============================================================
# StackSats imports
# ============================================================

try:
    from stacksats.runner.core import StrategyRunner, BacktestConfig
except ImportError:
    from stacksats.runner.core import StrategyRunner
    from stacksats.strategy_types import BacktestConfig

from stacksats.strategies.stable.mvrv.core import MVRVStrategy
from stacksats.strategies.stable.signals.momentum import MomentumStrategy
from stacksats.strategies.stable.baselines.uniform import UniformStrategy


# ============================================================
# Project imports
# ============================================================
# Import shared project modules through src, so the notebook can reuse:
# - data preparation utilities
# - shared config values
# - strategy_utils.run_year_by_year()
# - strategy_utils.compute_performance_summary()
# - plots.plot_strategy_full_period()

from src import config as src_config, data_utils, plots, strategy_utils

# Keep the original variable name used throughout existing notebook cells.
config = src_config

# Backward-compatible aliases used by existing cells below.
check_stacksats_data = data_utils.check_stacksats_data
export_one_year = strategy_utils.export_one_year


# ============================================================
# StackSats prepared dataset check
# ============================================================
# Important:
# Use the already-prepared StackSats file if it exists.
# Only call check_stacksats_data() if the prepared file is missing.
#
# This avoids failing just because data/raw/brk_metrics.parquet is missing,
# when ~/.stacksats/data/bitcoin_analytics.parquet already exists.

btc_path = config.STACKSATS_DATA_PATH

if btc_path.exists():
    print(f"Using existing prepared StackSats dataset: {btc_path}")
else:
    print(f"Prepared StackSats dataset not found at: {btc_path}")
    print("Trying to prepare it from raw BRK metrics...")

    if not check_stacksats_data(
        path=config.STACKSATS_DATA_PATH,
        raw_path=config.RAW_PATH,
    ):
        raise FileNotFoundError(
            "Could not prepare or load StackSats data.\n"
            f"Prepared path: {config.STACKSATS_DATA_PATH}\n"
            f"Raw path:      {config.RAW_PATH}\n\n"
            "If the prepared file already exists somewhere else, update "
            "STACKSATS_DATA_PATH in src/config.py."
        )

print(f"Final BTC dataset path: {btc_path}")


# ============================================================
# Strategy configuration
# ============================================================

# Budget used per calendar-year evaluation window.
TOTAL_BUDGET_USD = config.TOTAL_BUDGET_USD
SATS_PER_BTC = config.SATS_PER_BTC

# Train and test periods from shared config.
TRAIN_START = f"{config.TRAIN_START_YEAR}-01-01"
TRAIN_END = f"{config.SPLIT_YEAR - 1}-12-31"

TEST_START = f"{config.SPLIT_YEAR}-01-01"
TEST_END = f"{config.TEST_END_YEAR}-12-31"

# Minimum number of rows required for a valid calendar-year evaluation window.
WINDOW_SIZE = config.MIN_DAYS_PER_YEAR

# Default lookbacks used before grid search selects the best combination.
# These values are overwritten later by the best-performing grid result.
MOMENTUM_LOOKBACK = 60
SMA_LOOKBACK = 200
REGIME_LOOKBACK = 200

# Lookback grid values to test.
MOMENTUM_LOOKBACK_GRID = [
    21, 30, 45, 60, 75, 90, 120, 150, 180
]

SMA_LOOKBACK_GRID = [
    60, 90, 120, 150, 180, 200, 210, 240, 270, 300, 340
]

REGIME_LOOKBACK_GRID = [
    60, 90, 120, 150, 180, 200, 210, 240, 270, 300, 340
]

# Unique lookback values used to create rolling features.
LOOKBACK_DAYS = sorted({
    MOMENTUM_LOOKBACK,
    SMA_LOOKBACK,
    REGIME_LOOKBACK,
})

# Small floor to avoid zero or negative allocation signals.
SIGNAL_FLOOR = 1e-8

# Minimum number of days required for a regime to be evaluated.
MIN_REGIME_DAYS = 20

# Tolerance used to decide whether a result is better, worse, or tied.
STATUS_TOLERANCE_PCT = 1e-6

# Fallback strategy options used if a regime appears in test but was not seen in training.
FALLBACK_STRATEGY_GRID = [
    "sma",
    "stacksats_mvrv_weight",
    "stacksats_momentum_weight",
]


# ============================================================
# Regime-strategy helper configuration
# ============================================================

def resolve_fallback_strategy(fallback_strategy, sma_lookback=None):
    """Convert a fallback strategy option into the actual weight column name."""
    if sma_lookback is None:
        sma_lookback = SMA_LOOKBACK

    if fallback_strategy == "sma":
        return f"sma_{sma_lookback}d_weight"

    return fallback_strategy


def get_fallback_strategy_cols(sma_lookback=None):
    """Return actual fallback strategy weight columns for the selected SMA lookback."""
    if sma_lookback is None:
        sma_lookback = SMA_LOOKBACK

    return [
        resolve_fallback_strategy(fallback_strategy, sma_lookback)
        for fallback_strategy in FALLBACK_STRATEGY_GRID
    ]


def get_candidate_cols(sma_lookback=None):
    """Return candidate strategy columns for the selected SMA lookback."""
    if sma_lookback is None:
        sma_lookback = SMA_LOOKBACK

    return [
        "stacksats_mvrv_weight",
        "stacksats_momentum_weight",
        f"sma_{sma_lookback}d_weight",
    ]


# Default fallback strategy before grid search overwrites it.
FALLBACK_STRATEGY = resolve_fallback_strategy("sma", SMA_LOOKBACK)

CANDIDATE_COLS = get_candidate_cols(SMA_LOOKBACK)
FALLBACK_CANDIDATE_COLS = get_fallback_strategy_cols(SMA_LOOKBACK)


# ============================================================
# Final setup printout
# ============================================================

print("Train period:", TRAIN_START, "to", TRAIN_END)
print("Test period: ", TEST_START, "to", TEST_END)
print("Total budget USD:", TOTAL_BUDGET_USD)
print("SATS_PER_BTC:", SATS_PER_BTC)
print("Candidate columns:", CANDIDATE_COLS)
print("Fallback columns:", FALLBACK_CANDIDATE_COLS)

Current working directory: c:\Users\ragha\Documents\capstone\AdaptiveSats\notebooks\proposed_strategies
Project root: C:\Users\ragha\Documents\capstone\AdaptiveSats
src exists: True
Using existing prepared StackSats dataset: C:\Users\ragha\.stacksats\data\bitcoin_analytics.parquet
Final BTC dataset path: C:\Users\ragha\.stacksats\data\bitcoin_analytics.parquet
Train period: 2018-01-01 to 2023-12-31
Test period:  2024-01-01 to 2025-12-31
Total budget USD: 1000.0
SATS_PER_BTC: 100000000
Candidate columns: ['stacksats_mvrv_weight', 'stacksats_momentum_weight', 'sma_200d_weight']
Fallback columns: ['sma_200d_weight', 'stacksats_mvrv_weight', 'stacksats_momentum_weight']


In [2]:
# ============================================================
# Cell 2: Helper functions
# ============================================================
# This cell defines general helper functions used across the notebook.
# These functions are reused for labeling results, formatting chart text,
# trimming complete windows, normalizing weights, and summarizing results.

def get_status_from_pct_diff(pct_diff, tolerance=STATUS_TOLERANCE_PCT):
    """
    Convert percentage improvement into a simple status label.

    Parameters
    ----------
    pct_diff : float
        Percentage difference of strategy performance versus DCA.
    tolerance : float
        Small threshold used to avoid classifying tiny numerical differences
        as meaningful wins or losses.

    Returns
    -------
    str
        'better' if strategy beats DCA, 'worse' if it underperforms,
        and 'tie' if the difference is within tolerance.
    """

    # If improvement is greater than tolerance, strategy is better.
    if pct_diff > tolerance:
        return "better"

    # If improvement is below negative tolerance, strategy is worse.
    if pct_diff < -tolerance:
        return "worse"

    # Otherwise treat it as no meaningful difference.
    return "tie"


def get_calendar_year_windows(
    df: pd.DataFrame,
    split_start: str,
    split_end: str,
    min_days: int = WINDOW_SIZE,
):
    """
    Build one evaluation window per calendar year.

    This is used instead of fixed 365-row chunking so leap years are handled
    correctly. For example, the 2024 test window is 2024-01-01 to 2024-12-31
    with 366 rows, and the next test window starts on 2025-01-01.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe with a datetime `date` column.
    split_start : str
        Inclusive split start date, such as TRAIN_START or TEST_START.
    split_end : str
        Inclusive split end date, such as TRAIN_END or TEST_END.
    min_days : int
        Minimum rows required to keep a calendar-year window. The default is
        365, so normal years and leap years are both valid.

    Returns
    -------
    tuple[pd.DataFrame, list[dict], pd.DataFrame, pd.DataFrame]
        Raw split dataframe, list of calendar-year window dictionaries,
        metadata for kept windows, and metadata for skipped years.
    """

    split_start_ts = pd.to_datetime(split_start)
    split_end_ts = pd.to_datetime(split_end)

    raw_split = (
        df[
            (df["date"] >= split_start_ts) &
            (df["date"] <= split_end_ts)
        ]
        .copy()
        .sort_values("date")
        .reset_index(drop=True)
    )

    windows = []
    kept_rows = []
    skipped_rows = []

    for year in range(split_start_ts.year, split_end_ts.year + 1):
        year_start = max(pd.Timestamp(year=year, month=1, day=1), split_start_ts)
        year_end = min(pd.Timestamp(year=year, month=12, day=31), split_end_ts)

        year_df = (
            raw_split[
                (raw_split["date"] >= year_start) &
                (raw_split["date"] <= year_end)
            ]
            .copy()
            .sort_values("date")
            .reset_index(drop=True)
        )

        observed_days = len(year_df)
        expected_days = (year_end - year_start).days + 1

        if observed_days < min_days:
            skipped_rows.append({
                "year": year,
                "start_date": year_start,
                "end_date": year_end,
                "observed_days": observed_days,
                "expected_calendar_days": expected_days,
                "reason": f"less than {min_days} rows",
            })
            continue

        window_number = len(windows) + 1
        windows.append({
            "window": window_number,
            "year": year,
            "start_date": year_df["date"].min(),
            "end_date": year_df["date"].max(),
            "days": observed_days,
            "expected_calendar_days": expected_days,
            "data": year_df,
        })

        kept_rows.append({
            "window": window_number,
            "year": year,
            "start_date": year_df["date"].min(),
            "end_date": year_df["date"].max(),
            "days": observed_days,
            "expected_calendar_days": expected_days,
            "is_leap_window": observed_days == 366,
        })

    window_metadata_df = pd.DataFrame(kept_rows)
    skipped_metadata_df = pd.DataFrame(skipped_rows)

    return raw_split, windows, window_metadata_df, skipped_metadata_df


def concat_calendar_windows(windows):
    """Concatenate the kept calendar-year windows into one dataframe."""

    if not windows:
        return pd.DataFrame()

    return pd.concat(
        [w["data"].copy() for w in windows],
        ignore_index=True,
    )

def build_simple_normalized_weights(signal_multiplier, signal_floor=SIGNAL_FLOOR):
    """
    Convert signal multipliers into normalized daily allocation weights.

    Parameters
    ----------
    signal_multiplier : array-like
        Raw signal strength or multiplier values.
    signal_floor : float
        Minimum allowed signal value to prevent zero or negative weights.

    Returns
    -------
    np.ndarray
        Daily allocation weights that sum to 1.
    """

    # Convert input into a numpy array.
    signal_multiplier = np.asarray(signal_multiplier, dtype=float)

    # Prevent empty signal arrays.
    if len(signal_multiplier) == 0:
        raise ValueError("Empty signal array.")

    # Replace NaN and infinity values with 0.
    clean_signal = np.nan_to_num(
        signal_multiplier,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    # Apply a small floor so no value is exactly zero or negative.
    clean_signal = np.maximum(clean_signal, signal_floor)

    # If all signals are invalid or zero, fall back to uniform weights.
    if clean_signal.sum() <= 0:
        return np.full(len(clean_signal), 1.0 / len(clean_signal))

    # Normalize so all daily weights sum to 1.
    return clean_signal / clean_signal.sum()


def summarize_spd_like_composite(window_summary_df):
    """
    Summarize performance across all 365-day windows.

    Parameters
    ----------
    window_summary_df : pd.DataFrame
        Window-level summary dataframe containing strategy_sats, dca_sats,
        strategy_spd, dca_spd, and result columns.

    Returns
    -------
    dict
        Summary metrics including total sats, SPD sums, improvement percentage,
        wins, losses, ties, and win rate.
    """

    # Number of full 365-day windows.
    n_windows = len(window_summary_df)

    # Sum strategy and DCA sats-per-dollar across windows.
    strategy_spd_sum = window_summary_df["strategy_spd"].sum()
    dca_spd_sum = window_summary_df["dca_spd"].sum()

    # Extra sats-per-dollar gained or lost vs DCA.
    extra_spd_sum = strategy_spd_sum - dca_spd_sum

    # Ratio of strategy SPD to DCA SPD.
    spd_ratio = strategy_spd_sum / dca_spd_sum

    # Percentage improvement over DCA.
    improvement_pct = (spd_ratio - 1.0) * 100.0

    # Sum total sats accumulated by strategy and DCA.
    strategy_sats = window_summary_df["strategy_sats"].sum()
    dca_sats = window_summary_df["dca_sats"].sum()

    # Extra sats accumulated vs DCA.
    extra_sats = strategy_sats - dca_sats

    # Count windows where strategy beat, lost to, or tied DCA.
    wins = int((window_summary_df["result"] == "better").sum())
    losses = int((window_summary_df["result"] == "worse").sum())
    ties = int((window_summary_df["result"] == "tie").sum())

    # Window win rate.
    win_rate_pct = wins / n_windows * 100.0 if n_windows > 0 else 0.0

    # Return one summary dictionary.
    return {
        "n_windows": n_windows,
        "wins": wins,
        "losses": losses,
        "ties": ties,
        "win_rate_pct": win_rate_pct,

        "strategy_sats": strategy_sats,
        "dca_sats": dca_sats,
        "extra_sats_vs_dca": extra_sats,

        "strategy_spd_sum": strategy_spd_sum,
        "dca_spd_sum": dca_spd_sum,
        "extra_spd_sum_vs_dca": extra_spd_sum,

        "strategy_spd_avg": strategy_spd_sum / n_windows,
        "dca_spd_avg": dca_spd_sum / n_windows,
        "extra_spd_avg_vs_dca": extra_spd_sum / n_windows,

        "spd_ratio": spd_ratio,
        "improvement_pct": improvement_pct,
    }


In [3]:
# ============================================================
# Cell 3: Load BTC data
# ============================================================
# This cell loads the prepared Bitcoin analytics parquet file.
# It checks that all required columns exist before moving forward.

if not btc_path.exists():
    raise FileNotFoundError(f"Could not find: {btc_path}")

btc_df = (
    pl.read_parquet(config.STACKSATS_DATA_PATH)
    .with_columns(pl.col("date").cast(pl.Datetime))
    .sort("date")
)

required_cols = [
    "date",
    "price_usd",
    "mvrv",
    "realized_cap_growth_rate",
    "market_cap_growth_rate",
]

missing_cols = [col for col in required_cols if col not in btc_df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns from bitcoin_analytics.parquet: {missing_cols}")

print("Loaded BTC rows:", btc_df.height)

display(
    btc_df.select(
        pl.col("date").min().alias("min_date"),
        pl.col("date").max().alias("max_date"),
    )
)


Loaded BTC rows: 5689


min_date,max_date
datetime[μs],datetime[μs]
2010-08-16 00:00:00,2026-03-13 00:00:00


In [4]:
# ============================================================
# Cell 4: StackSats strategy export setup
# ============================================================
# This cell sets up the StackSats strategy runner and strategy objects.
# The export function below uses src.strategy_utils.export_one_year().

runner = StrategyRunner()

stacksats_strategy_objects = {
    "stacksats_mvrv_weight": MVRVStrategy(),
    "stacksats_momentum_weight": MomentumStrategy(),
    "stacksats_uniform_weight": UniformStrategy(),
}

# Cache prevents recomputing weights for the same strategy/window repeatedly.
_export_cache = {}


def export_stacksats_weight_frame_for_window(
    strategy_key: str,
    window_df: pd.DataFrame,
    full_btc_df: pl.DataFrame,
):
    """
    Export StackSats strategy weights for one calendar-year window.

    Refactored:
    - Uses src.strategy_utils.export_one_year()
    - Keeps the same output expected by the rest of this notebook:
      columns: date, raw_weight
    """

    if window_df.empty:
        raise ValueError("window_df is empty.")

    window_df = window_df.copy()
    window_df["date"] = pd.to_datetime(window_df["date"]).dt.normalize()

    window_years = sorted(window_df["date"].dt.year.unique())

    if len(window_years) != 1:
        raise ValueError(
            "export_one_year() expects one calendar-year window. "
            f"Found years: {window_years}"
        )

    year = int(window_years[0])
    window_start = window_df["date"].min()
    window_end = window_df["date"].max()

    cache_key = (
        strategy_key,
        window_start.strftime("%Y-%m-%d"),
        window_end.strftime("%Y-%m-%d"),
        "export_one_year",
    )

    if cache_key in _export_cache:
        return _export_cache[cache_key].copy()

    if strategy_key not in stacksats_strategy_objects:
        raise KeyError(
            f"Unknown strategy_key: {strategy_key}. "
            f"Available keys: {list(stacksats_strategy_objects.keys())}"
        )

    strategy = stacksats_strategy_objects[strategy_key]

    exported_df = export_one_year(
        strategy=strategy,
        btc_data=full_btc_df,
        year=year,
        runner=runner,
    )

    if exported_df is None:
        raise ValueError(
            f"export_one_year returned None for {strategy_key}, year {year}"
        )

    if not isinstance(exported_df, pl.DataFrame):
        exported_df = pl.from_pandas(exported_df)

    if exported_df.is_empty():
        raise ValueError(
            f"export_one_year returned no weights for {strategy_key}, year {year}"
        )

    latest_window_weights = (
        exported_df
        .select(["date", "weight"])
        .sort("date")
        .rename({"weight": "raw_weight"})
        .to_pandas()
    )

    latest_window_weights["date"] = pd.to_datetime(
        latest_window_weights["date"]
    ).dt.normalize()

    latest_window_weights = latest_window_weights[
        (latest_window_weights["date"] >= window_start)
        & (latest_window_weights["date"] <= window_end)
    ].copy()

    latest_window_weights = (
        latest_window_weights
        .sort_values("date")
        .drop_duplicates(subset=["date"], keep="last")
        .reset_index(drop=True)
    )

    if latest_window_weights.empty:
        raise ValueError(
            f"No usable weights for {strategy_key} from "
            f"{window_start.date()} to {window_end.date()}"
        )

    _export_cache[cache_key] = latest_window_weights.copy()

    return latest_window_weights.copy()


In [5]:
# ============================================================
# Cell 5: Feature engineering and simplified regime classification
# ============================================================
# This cell creates reusable functions for:
# 1. engineering rolling lookback features
# 2. assigning the simpler combined regime
# 3. preparing a clean dataframe for any lookback combination

def classify_btc_mvrv_market_cap_regime(
    row,
    momentum_lookback=None,
    sma_lookback=None,
    regime_lookback=None,
):
    """
    Classify each day into a simpler BTC/on-chain regime.

    The regime combines:
    1. BTC trend regime using SMA ratio and returns
    2. MVRV valuation regime
    3. market-cap / realized-cap growth regime

    Returns
    -------
    str
        Combined regime label in the format:
        BTC trend regime | MVRV valuation regime | cap-growth regime
    """

    if momentum_lookback is None:
        momentum_lookback = MOMENTUM_LOOKBACK
    if sma_lookback is None:
        sma_lookback = SMA_LOOKBACK
    if regime_lookback is None:
        regime_lookback = REGIME_LOOKBACK

    # ------------------------------------------------------------
    # BTC trend features
    # ------------------------------------------------------------
    sma_selected_ratio = row[f"price_{sma_lookback}d_sma_ratio"]
    sma_regime_ratio = row[f"price_{regime_lookback}d_sma_ratio"]
    return_momentum = row[f"btc_return_{momentum_lookback}d"]
    return_sma = row[f"btc_return_{sma_lookback}d"]

    # ------------------------------------------------------------
    # On-chain / market features
    # ------------------------------------------------------------
    mvrv = row["mvrv"]
    realized_growth = row["realized_cap_growth_rate"]
    market_growth = row["market_cap_growth_rate"]

    # ------------------------------------------------------------
    # 1. BTC trend regime without drawdown
    # ------------------------------------------------------------
    if sma_regime_ratio >= 1.05 and return_sma > 0:
        btc_regime = "BTC Bull"

    elif sma_selected_ratio <= 0.95 and return_sma < 0:
        btc_regime = "BTC Bear"

    elif sma_selected_ratio < 1.0 and return_momentum > 0:
        btc_regime = "BTC Recovery"

    else:
        btc_regime = "BTC Neutral"

    # ------------------------------------------------------------
    # 2. MVRV valuation regime
    # ------------------------------------------------------------
    if mvrv < 1.0:
        valuation_regime = "Low MVRV"

    elif mvrv > 2.5:
        valuation_regime = "High MVRV"

    else:
        valuation_regime = "Normal MVRV"

    # ------------------------------------------------------------
    # 3. Market-cap / realized-cap growth regime
    # ------------------------------------------------------------
    if realized_growth > market_growth:
        cap_regime = "Realized Growth Leading"

    else:
        cap_regime = "Market Growth Leading"

    # ------------------------------------------------------------
    # Final combined regime
    # ------------------------------------------------------------
    return btc_regime + " | " + valuation_regime + " | " + cap_regime




def prepare_btc_data_for_lookbacks(
    full_btc_df,
    momentum_lookback,
    sma_lookback,
    regime_lookback,
):
    """
    Create rolling features and simplified regimes for one lookback combination.
    """

    lookback_days = sorted({
        momentum_lookback,
        sma_lookback,
        regime_lookback,
    })

    feature_exprs = []

    for d in lookback_days:
        feature_exprs.extend([
            pl.col("price_usd")
            .rolling_mean(window_size=d, min_samples=max(3, int(d * 0.30)))
            .alias(f"price_{d}d_sma"),

            pl.col("price_usd")
            .pct_change(d)
            .alias(f"btc_return_{d}d"),
        ])

    temp_btc_df = full_btc_df.with_columns(feature_exprs)

    ratio_exprs = []

    for d in lookback_days:
        ratio_exprs.append(
            (pl.col("price_usd") / pl.col(f"price_{d}d_sma"))
            .alias(f"price_{d}d_sma_ratio")
        )

    temp_btc_df = temp_btc_df.with_columns(ratio_exprs)

    temp_btc_data = temp_btc_df.to_pandas()
    temp_btc_data["date"] = pd.to_datetime(temp_btc_data["date"])

    feature_cols = [
        "price_usd",
        "mvrv",
        "realized_cap_growth_rate",
        "market_cap_growth_rate",
    ]

    for d in lookback_days:
        feature_cols.extend([
            f"price_{d}d_sma",
            f"price_{d}d_sma_ratio",
            f"btc_return_{d}d",
        ])

    temp_btc_data = (
        temp_btc_data
        .dropna(subset=feature_cols)
        .sort_values("date")
        .reset_index(drop=True)
    )

    temp_btc_data["combined_regime"] = temp_btc_data.apply(
        lambda row: classify_btc_mvrv_market_cap_regime(
            row,
            momentum_lookback=momentum_lookback,
            sma_lookback=sma_lookback,
            regime_lookback=regime_lookback,
        ),
        axis=1,
    )

    return temp_btc_data


In [6]:
# ============================================================
# Cell 6: Candidate strategy weights
# ============================================================
# This cell creates daily weights for each candidate strategy.
# Candidate strategies are StackSats MVRV, StackSats Momentum, and custom SMA.
#
# Important:
# StackSats strategies keep the latest export window for each calendar year.
# In leap years, that may be 365 rows instead of 366. To keep all strategies
# comparable, this function evaluates DCA, SMA, MVRV, and Momentum on the
# common exported dates for that year.

def create_candidate_strategy_weights_simple(data, sma_lookback=None):
    """
    Create daily allocation weights for all candidate strategies.

    Parameters
    ----------
    data : pd.DataFrame
        One calendar-year window of BTC data with regime and engineered features.
    sma_lookback : int or None
        SMA lookback used for the custom SMA candidate.

    Returns
    -------
    pd.DataFrame
        Original data plus strategy weight columns:
        - dca_weight
        - stacksats_mvrv_weight
        - stacksats_momentum_weight
        - sma_{sma_lookback}d_weight
    """

    if sma_lookback is None:
        sma_lookback = SMA_LOOKBACK

    # Sort data by date and reset index.
    df = data.copy().sort_values("date").reset_index(drop=True)
    df["date"] = pd.to_datetime(df["date"])

    # Stop if the input window is empty.
    if len(df) == 0:
        raise ValueError("No data available.")

    # Export StackSats MVRV latest-window weights for this calendar year.
    mvrv_weights = export_stacksats_weight_frame_for_window(
        strategy_key="stacksats_mvrv_weight",
        window_df=df,
        full_btc_df=btc_df,
    ).rename(columns={"raw_weight": "stacksats_mvrv_raw_weight"})

    # Export StackSats Momentum latest-window weights for this calendar year.
    momentum_weights = export_stacksats_weight_frame_for_window(
        strategy_key="stacksats_momentum_weight",
        window_df=df,
        full_btc_df=btc_df,
    ).rename(columns={"raw_weight": "stacksats_momentum_raw_weight"})

    # Export StackSats UniformStrategy latest-window weights for this calendar year.
    uniform_weights = export_stacksats_weight_frame_for_window(
        strategy_key="stacksats_uniform_weight",
        window_df=df,
        full_btc_df=btc_df,
    ).rename(columns={"raw_weight": "stacksats_uniform_raw_weight"})

    # Keep only the common dates available in all StackSats exports.
    df = (
        df
        .merge(mvrv_weights, on="date", how="inner")
        .merge(momentum_weights, on="date", how="inner")
        .merge(uniform_weights, on="date", how="inner")
        .sort_values("date")
        .reset_index(drop=True)
    )

    # Number of usable days in the current calendar-year export.
    n = len(df)
    if n < WINDOW_SIZE:
        raise ValueError(
            f"Only {n} common StackSats export dates were available; "
            f"expected at least {WINDOW_SIZE}."
        )

    # Uniform DCA benchmark from StackSats UniformStrategy.
    # Normalize the exported raw uniform weights over the same common dates
    df["dca_weight"] = build_simple_normalized_weights(
        df["stacksats_uniform_raw_weight"].values
    )

    # Normalize StackSats raw weights so each candidate sums to 1 over the
    # same evaluated dates.
    df["stacksats_mvrv_weight"] = build_simple_normalized_weights(
        df["stacksats_mvrv_raw_weight"].values
    )

    df["stacksats_momentum_weight"] = build_simple_normalized_weights(
        df["stacksats_momentum_raw_weight"].values
    )

    # Candidate 3: Custom SMA strategy.
    # price/SMA ratio < 1 means BTC price is below the selected SMA.
    # In that case, sma_signal becomes positive and allocation increases.
    sma_signal = (1.0 - df[f"price_{sma_lookback}d_sma_ratio"]).clip(-1, 1)

    # Convert SMA signal into a multiplier.
    # 1.50 controls how strongly the SMA signal changes allocation.
    sma_multiplier = np.maximum(SIGNAL_FLOOR, 1.0 + 1.50 * sma_signal)

    # Normalize SMA multipliers into daily weights that sum to 1 over the
    # same dates used by StackSats exports.
    df[f"sma_{sma_lookback}d_weight"] = build_simple_normalized_weights(
        sma_multiplier.values
    )

    # Keep raw export columns for debugging, but they are not used for strategy selection.
    return df


In [7]:
# ============================================================
# Cell 7: Calendar-year evaluation functions for one lookback combination
# ============================================================
# This cell evaluates candidate strategies by regime, learns the best mapping,
# and applies that mapping to train/test calendar-year windows.
# - Candidate strategy evaluation uses previous-day shifted weights.
# - This makes the training regime-mapping step consistent with the final
#   application step, where today's allocation uses yesterday's signal.
#
# Leap-year handling:
# - Calendar years are created separately for train and test.
# - For example, 2024 is requested as 2024-01-01 to 2024-12-31.
# - Because StackSats may return rolling 365-day export windows, the latest
#   export window inside a leap year may contain 365 rows, such as
#   2024-01-02 to 2024-12-31.
# - The notebook keeps the export rows with the latest end_date.


def evaluate_strategies_by_regime_in_365_windows(
    windows,
    total_budget_usd=TOTAL_BUDGET_USD,
    sma_lookback=None,
):
    """
    Evaluate all candidate strategies inside each regime for every calendar-year window.

    Input is a list of calendar-year windows produced by get_calendar_year_windows().

    Important:
    - Candidate strategy weights are shifted by one day before evaluation.
    - This avoids learning the regime-to-strategy mapping from same-day signals.
    - The shifted candidate weights are normalized over the full calendar-year
      window so each candidate still spends the full yearly budget.
    """

    if sma_lookback is None:
        sma_lookback = SMA_LOOKBACK

    candidate_cols = get_candidate_cols(sma_lookback)
    rows = []

    for window_info in windows:
        window_idx = window_info["window"]
        year = window_info["year"]
        window_data = window_info["data"].copy().reset_index(drop=True)

        df = create_candidate_strategy_weights_simple(
            data=window_data,
            sma_lookback=sma_lookback,
        )

        if df.empty:
            raise ValueError(
                f"No candidate strategy weights were created for year {year}."
            )

        # Original candidate columns are same-day weights.
        # For learning the best regime mapping, use previous-day shifted
        # candidate weights so training evaluation matches final application.
        #
        # Example:
        #   today's MVRV allocation uses yesterday's MVRV weight.
        #
        # The first day has no previous-day signal, so use a neutral equal
        # weight. Then normalize so each shifted candidate sums to 1 over
        # the full evaluated calendar-year window.
        # ------------------------------------------------------------

        shifted_candidate_cols = {}

        for col in candidate_cols:
            shifted_col = f"{col}_shifted"

            df[shifted_col] = df[col].shift(1)

            neutral_first_day_weight = 1.0 / len(df)
            df[shifted_col] = df[shifted_col].fillna(neutral_first_day_weight)

            shifted_sum = df[shifted_col].sum()

            if shifted_sum <= 0 or pd.isna(shifted_sum):
                df[shifted_col] = 1.0 / len(df)
            else:
                df[shifted_col] = df[shifted_col] / shifted_sum

            shifted_candidate_cols[col] = shifted_col

        # ------------------------------------------------------------
        # Regime-level candidate evaluation
        # ------------------------------------------------------------

        for regime, regime_df in df.groupby("combined_regime"):
            if len(regime_df) < MIN_REGIME_DAYS:
                continue

            # DCA benchmark remains the same yearly DCA allocation.
            dca_sats = (
                regime_df["dca_weight"]
                * total_budget_usd
                / regime_df["price_usd"]
                * SATS_PER_BTC
            ).sum()

            dca_spd = dca_sats / total_budget_usd

            for col in candidate_cols:
                shifted_col = shifted_candidate_cols[col]

                strategy_sats = (
                    regime_df[shifted_col]
                    * total_budget_usd
                    / regime_df["price_usd"]
                    * SATS_PER_BTC
                ).sum()

                strategy_spd = strategy_sats / total_budget_usd
                extra_sats = strategy_sats - dca_sats
                extra_spd = strategy_spd - dca_spd

                spd_ratio = strategy_spd / dca_spd
                improvement_pct = (spd_ratio - 1.0) * 100.0

                rows.append({
                    "train_window": window_idx,
                    "year": year,
                    "window_start_date": window_info["start_date"],
                    "window_end_date": window_info["end_date"],
                    "window_days": window_info["days"],
                    "combined_regime": regime,
                    "days": len(regime_df),
                    "strategy": col,

                    # These are based on shifted candidate weights.
                    "strategy_sats": strategy_sats,
                    "dca_sats": dca_sats,
                    "extra_sats_vs_dca": extra_sats,
                    "strategy_spd": strategy_spd,
                    "dca_spd": dca_spd,
                    "extra_spd_vs_dca": extra_spd,
                    "spd_ratio": spd_ratio,
                    "improvement_pct": improvement_pct,
                    "status": get_status_from_pct_diff(improvement_pct),

                    # Debug/provenance columns.
                    "candidate_weight_col": col,
                    "candidate_weight_col_used_for_eval": shifted_col,
                    "uses_shifted_candidate_weight": True,
                })

    return pd.DataFrame(rows)


def learn_best_mapping_by_regime(train_regime_results_df):
    """
    Learn the best candidate strategy for each regime using training data only.
    """

    best_mapping_df = (
        train_regime_results_df
        .groupby(["combined_regime", "strategy"], as_index=False)
        .agg(
            total_days=("days", "sum"),
            mean_improvement_pct=("improvement_pct", "mean"),
            median_improvement_pct=("improvement_pct", "median"),
            mean_extra_spd_vs_dca=("extra_spd_vs_dca", "mean"),
            total_extra_sats_vs_dca=("extra_sats_vs_dca", "sum"),
            windows_seen=("train_window", "nunique"),
        )
    )

    best_mapping_df = (
        best_mapping_df
        .sort_values(
            ["combined_regime", "mean_improvement_pct", "total_days"],
            ascending=[True, False, False],
        )
        .groupby("combined_regime")
        .head(1)
        .reset_index(drop=True)
    )

    best_mapping_df["status"] = best_mapping_df["mean_improvement_pct"].apply(
        get_status_from_pct_diff
    )

    return best_mapping_df


def apply_regime_mapping_to_one_window(
    window_data,
    best_mapping_df,
    total_budget_usd=TOTAL_BUDGET_USD,
    window_number=None,
    year=None,
    sma_lookback=None,
    fallback_strategy=None,
):
    """
    Apply the learned regime mapping to one calendar-year window.

    Important:
    - The regime-to-strategy mapping is already learned from training data.
    - This function applies that mapping to the given window.
    - To avoid same-day look-ahead bias, today's allocation uses yesterday's
      selected strategy weight.
    """

    if sma_lookback is None:
        sma_lookback = SMA_LOOKBACK

    if fallback_strategy is None:
        fallback_strategy = FALLBACK_STRATEGY

    fallback_strategy = resolve_fallback_strategy(
        fallback_strategy=fallback_strategy,
        sma_lookback=sma_lookback,
    )

    df = create_candidate_strategy_weights_simple(
        data=window_data,
        sma_lookback=sma_lookback,
    )

    regime_to_strategy = dict(
        zip(best_mapping_df["combined_regime"], best_mapping_df["strategy"])
    )

    mapped_strategy = df["combined_regime"].map(regime_to_strategy)

    df["used_fallback_strategy"] = mapped_strategy.isna()
    df["fallback_strategy"] = fallback_strategy
    df["selected_strategy"] = mapped_strategy.fillna(fallback_strategy)

    # ------------------------------------------------------------
    # Selected strategy raw weight
    # ------------------------------------------------------------
    # This is the same-day raw weight suggested by the selected strategy.
    # We keep it for debugging / comparison, but we do not directly use it
    # for same-day allocation.
    df["raw_selected_weight_unshifted"] = df.apply(
        lambda row: row[row["selected_strategy"]],
        axis=1,
    )

    # ------------------------------------------------------------
    # Avoid same-day look-ahead bias
    # ------------------------------------------------------------
    # Today's allocation uses yesterday's selected strategy weight.
    # This avoids using today's finalized price/on-chain features to decide
    # today's buy.
    df["raw_selected_weight"] = df["raw_selected_weight_unshifted"].shift(1)

    # First day has no previous-day signal.
    # Use a neutral equal-weight value instead of using the DCA benchmark column.
    neutral_first_day_weight = 1.0 / len(df)

    df["raw_selected_weight"] = df["raw_selected_weight"].fillna(
        neutral_first_day_weight
    )

    # ------------------------------------------------------------
    # Normalize shifted weights
    # ------------------------------------------------------------
    # This keeps the full yearly/window budget allocated.
    raw_weight_sum = df["raw_selected_weight"].sum()

    if raw_weight_sum <= 0 or pd.isna(raw_weight_sum):
        df["final_strategy_weight"] = 1.0 / len(df)
    else:
        df["final_strategy_weight"] = df["raw_selected_weight"] / raw_weight_sum

    # ------------------------------------------------------------
    # Dollar allocation
    # ------------------------------------------------------------
    df["final_strategy_usd"] = df["final_strategy_weight"] * total_budget_usd
    df["dca_usd"] = df["dca_weight"] * total_budget_usd

    # ------------------------------------------------------------
    # BTC / sats accumulation
    # ------------------------------------------------------------
    df["btc_accum_strategy"] = df["final_strategy_usd"] / df["price_usd"]
    df["btc_accum_dca"] = df["dca_usd"] / df["price_usd"]

    df["sats_accum_strategy"] = df["btc_accum_strategy"] * SATS_PER_BTC
    df["sats_accum_dca"] = df["btc_accum_dca"] * SATS_PER_BTC

    df["strategy_spd_daily"] = df["sats_accum_strategy"] / total_budget_usd
    df["dca_spd_daily"] = df["sats_accum_dca"] / total_budget_usd

    # ------------------------------------------------------------
    # Metadata
    # ------------------------------------------------------------
    if window_number is not None:
        df["window"] = window_number

    if year is not None:
        df["year"] = year

    return df


def apply_regime_mapping_to_window_set(
    windows,
    best_mapping_df,
    total_budget_usd=TOTAL_BUDGET_USD,
    sma_lookback=None,
    fallback_strategy=None,
):
    """
    Apply the learned regime mapping to every calendar-year window.
    """

    if sma_lookback is None:
        sma_lookback = SMA_LOOKBACK

    if fallback_strategy is None:
        fallback_strategy = FALLBACK_STRATEGY

    fallback_strategy = resolve_fallback_strategy(
        fallback_strategy=fallback_strategy,
        sma_lookback=sma_lookback,
    )

    window_dfs = []
    window_summary_rows = []

    for window_info in windows:
        window_strategy_df = apply_regime_mapping_to_one_window(
            window_data=window_info["data"].copy().reset_index(drop=True),
            best_mapping_df=best_mapping_df,
            total_budget_usd=total_budget_usd,
            window_number=window_info["window"],
            year=window_info["year"],
            sma_lookback=sma_lookback,
            fallback_strategy=fallback_strategy,
        )

        strategy_sats = window_strategy_df["sats_accum_strategy"].sum()
        dca_sats = window_strategy_df["sats_accum_dca"].sum()

        strategy_spd = strategy_sats / total_budget_usd
        dca_spd = dca_sats / total_budget_usd

        extra_sats = strategy_sats - dca_sats
        extra_spd = strategy_spd - dca_spd

        spd_ratio = strategy_spd / dca_spd
        improvement_pct = (spd_ratio - 1.0) * 100.0

        window_summary_rows.append({
            "window": window_info["window"],
            "year": window_info["year"],
            "start_date": window_strategy_df["date"].min(),
            "end_date": window_strategy_df["date"].max(),
            "days": len(window_strategy_df),
            "expected_calendar_days": window_info["expected_calendar_days"],
            "is_leap_window": len(window_strategy_df) == 366,
            "budget_usd": total_budget_usd,
            "strategy_sats": strategy_sats,
            "dca_sats": dca_sats,
            "extra_sats_vs_dca": extra_sats,
            "strategy_spd": strategy_spd,
            "dca_spd": dca_spd,
            "extra_spd_vs_dca": extra_spd,
            "spd_ratio": spd_ratio,
            "improvement_pct": improvement_pct,
            "result": get_status_from_pct_diff(improvement_pct),
            "weight_sum": window_strategy_df["final_strategy_weight"].sum(),
            "max_weight": window_strategy_df["final_strategy_weight"].max(),
            "min_weight": window_strategy_df["final_strategy_weight"].min(),
            "days_above_dca_weight": int(
                (window_strategy_df["final_strategy_weight"] > window_strategy_df["dca_weight"]).sum()
            ),
            "fallback_strategy": fallback_strategy,
            "fallback_days": int(window_strategy_df["used_fallback_strategy"].sum()),
        })

        window_dfs.append(window_strategy_df)

    if not window_dfs:
        raise ValueError("No valid calendar-year windows were available.")

    out_df = pd.concat(window_dfs, ignore_index=True)
    summary_df = pd.DataFrame(window_summary_rows)

    return out_df, summary_df


In [8]:
# ============================================================
# Cell 8: Run calendar-year strategy for one lookback combination
# ============================================================
# This function is used in two modes:
#
# 1. Grid-search mode:
#    include_test=False
#    - learns regime mapping from TRAIN only
#    - evaluates TRAIN only
#    - does NOT calculate test metrics
#
# 2. Final selected-combo mode:
#    include_test=True
#    - learns regime mapping from TRAIN only
#    - evaluates TRAIN
#    - evaluates TEST once after the best train-selected combo is fixed

def run_full_regime_strategy_for_lookbacks(
    momentum_lookback,
    sma_lookback,
    regime_lookback,
    fallback_strategy=None,
    total_budget_usd=TOTAL_BUDGET_USD,
    include_test=False,
):
    """
    Run the complete simple-regime strategy for one lookback combination.

    Important:
    - The best regime-to-strategy mapping is always learned from TRAIN data only.
    - During grid search, set include_test=False to avoid test peeking.
    - After the best combo is selected using TRAIN metrics, set include_test=True
      to calculate TEST performance once.
    """

    if fallback_strategy is None:
        fallback_strategy = FALLBACK_STRATEGY

    fallback_strategy = resolve_fallback_strategy(
        fallback_strategy=fallback_strategy,
        sma_lookback=sma_lookback,
    )

    # ------------------------------------------------------------
    # Prepare features and simplified regime labels
    # ------------------------------------------------------------
    local_btc_data = prepare_btc_data_for_lookbacks(
        full_btc_df=btc_df,
        momentum_lookback=momentum_lookback,
        sma_lookback=sma_lookback,
        regime_lookback=regime_lookback,
    )

    # ------------------------------------------------------------
    # TRAIN split only
    # ------------------------------------------------------------
    raw_train, local_train_windows, train_calendar_windows_df, train_skipped_years_df = get_calendar_year_windows(
        local_btc_data,
        TRAIN_START,
        TRAIN_END,
        min_days=WINDOW_SIZE,
    )

    local_train_eval_df = concat_calendar_windows(local_train_windows)
    local_n_train_windows = len(local_train_windows)

    if local_n_train_windows == 0:
        raise ValueError(
            "Not enough data to create train calendar-year windows "
            "for this lookback combination."
        )

    # ------------------------------------------------------------
    # Learn regime mapping from TRAIN only
    # ------------------------------------------------------------
    local_train_regime_results_df = evaluate_strategies_by_regime_in_365_windows(
        local_train_windows,
        total_budget_usd=total_budget_usd,
        sma_lookback=sma_lookback,
    )

    if local_train_regime_results_df.empty:
        raise ValueError("No regime-level training results were created.")

    local_best_mapping_df = learn_best_mapping_by_regime(
        local_train_regime_results_df
    )

    # ------------------------------------------------------------
    # Apply learned mapping to TRAIN
    # ------------------------------------------------------------
    local_train_strategy_df, local_train_window_summary_df = apply_regime_mapping_to_window_set(
        local_train_windows,
        local_best_mapping_df,
        total_budget_usd=total_budget_usd,
        sma_lookback=sma_lookback,
        fallback_strategy=fallback_strategy,
    )

    local_train_spd_summary = summarize_spd_like_composite(
        local_train_window_summary_df
    )

    # ------------------------------------------------------------
    # Build TRAIN split summary
    # ------------------------------------------------------------
    split_summary_rows = [
        {
            "split": "train",
            "start_date": raw_train["date"].min(),
            "end_date": raw_train["date"].max(),
            "rows_total": len(raw_train),
            "rows_eval": len(local_train_eval_df),
            "windows": local_n_train_windows,
            "calendar_window_start_dates": ", ".join(
                train_calendar_windows_df["start_date"].dt.strftime("%Y-%m-%d")
            ),
            "calendar_window_end_dates": ", ".join(
                train_calendar_windows_df["end_date"].dt.strftime("%Y-%m-%d")
            ),
            "skipped_years": (
                ", ".join(train_skipped_years_df["year"].astype(str))
                if not train_skipped_years_df.empty
                else ""
            ),
            "budget_rule": "$1,000 per calendar-year training window; StackSats uses latest export window per year",
        }
    ]

    # ------------------------------------------------------------
    # Default TEST placeholders for grid-search mode
    # ------------------------------------------------------------
    raw_test = pd.DataFrame()
    local_test_windows = []
    test_calendar_windows_df = pd.DataFrame()
    test_skipped_years_df = pd.DataFrame()
    local_test_eval_df = pd.DataFrame()
    local_test_strategy_df = pd.DataFrame()
    local_test_window_summary_df = pd.DataFrame()
    local_test_spd_summary = {}
    local_n_test_windows = 0

    # ------------------------------------------------------------
    # TEST evaluation only after best combo is fixed
    # ------------------------------------------------------------
    if include_test:
        raw_test, local_test_windows, test_calendar_windows_df, test_skipped_years_df = get_calendar_year_windows(
            local_btc_data,
            TEST_START,
            TEST_END,
            min_days=WINDOW_SIZE,
        )

        local_test_eval_df = concat_calendar_windows(local_test_windows)
        local_n_test_windows = len(local_test_windows)

        if local_n_test_windows == 0:
            raise ValueError(
                "Not enough data to create test calendar-year windows "
                "for this selected lookback combination."
            )

        local_test_strategy_df, local_test_window_summary_df = apply_regime_mapping_to_window_set(
            local_test_windows,
            local_best_mapping_df,
            total_budget_usd=total_budget_usd,
            sma_lookback=sma_lookback,
            fallback_strategy=fallback_strategy,
        )

        local_test_spd_summary = summarize_spd_like_composite(
            local_test_window_summary_df
        )

        split_summary_rows.append(
            {
                "split": "test",
                "start_date": raw_test["date"].min(),
                "end_date": raw_test["date"].max(),
                "rows_total": len(raw_test),
                "rows_eval": len(local_test_eval_df),
                "windows": local_n_test_windows,
                "calendar_window_start_dates": ", ".join(
                    test_calendar_windows_df["start_date"].dt.strftime("%Y-%m-%d")
                ),
                "calendar_window_end_dates": ", ".join(
                    test_calendar_windows_df["end_date"].dt.strftime("%Y-%m-%d")
                ),
                "skipped_years": (
                    ", ".join(test_skipped_years_df["year"].astype(str))
                    if not test_skipped_years_df.empty
                    else ""
                ),
                "budget_rule": "$1,000 per calendar-year test window; StackSats uses latest export window per year",
            }
        )

    local_split_summary_df = pd.DataFrame(split_summary_rows)

    return {
        "momentum_lookback": momentum_lookback,
        "sma_lookback": sma_lookback,
        "regime_lookback": regime_lookback,
        "fallback_strategy": fallback_strategy,
        "include_test": include_test,

        "btc_data": local_btc_data,
        "split_summary_df": local_split_summary_df,

        "train_calendar_windows_df": train_calendar_windows_df,
        "test_calendar_windows_df": test_calendar_windows_df,

        "train_skipped_years_df": train_skipped_years_df,
        "test_skipped_years_df": test_skipped_years_df,

        "train_eval_df": local_train_eval_df,
        "test_eval_df": local_test_eval_df,

        "train_regime_results_df": local_train_regime_results_df,
        "best_mapping_df": local_best_mapping_df,

        "train_strategy_df": local_train_strategy_df,
        "test_strategy_df": local_test_strategy_df,

        "train_window_summary_df": local_train_window_summary_df,
        "test_window_summary_df": local_test_window_summary_df,

        "train_spd_summary": local_train_spd_summary,
        "test_spd_summary": local_test_spd_summary,
    }

In [9]:
# ============================================================
# Cell 9: Train-only lookback + fallback grid search and final selected-combo test
# ============================================================
# This cell tests all combinations of MOMENTUM_LOOKBACK_GRID,
# SMA_LOOKBACK_GRID, REGIME_LOOKBACK_GRID, and FALLBACK_STRATEGY_GRID.
#
# Important:
# - Grid search uses TRAIN metrics only.
# - Test metrics are NOT calculated during grid search.
# - After the best combo is selected using TRAIN metrics, the selected combo
#   is run once with include_test=True.

lookback_grid_rows = []
lookback_grid_artifacts = {}

for momentum_lb, sma_lb, regime_lb in product(
    MOMENTUM_LOOKBACK_GRID,
    SMA_LOOKBACK_GRID,
    REGIME_LOOKBACK_GRID,
):
    fallback_strategy_cols = get_fallback_strategy_cols(sma_lb)

    for fallback_col in fallback_strategy_cols:
        combo_key = (momentum_lb, sma_lb, regime_lb, fallback_col)

        try:
            # ------------------------------------------------------------
            # TRAIN-ONLY grid search
            # ------------------------------------------------------------
            result = run_full_regime_strategy_for_lookbacks(
                momentum_lookback=momentum_lb,
                sma_lookback=sma_lb,
                regime_lookback=regime_lb,
                fallback_strategy=fallback_col,
                total_budget_usd=TOTAL_BUDGET_USD,
                include_test=False,
            )

            lookback_grid_artifacts[combo_key] = result

            train_summary = result["train_spd_summary"]

            train_fallback_days = int(
                result["train_strategy_df"]["used_fallback_strategy"].sum()
            )

            lookback_grid_rows.append({
                "status": "success",
                "momentum_lookback": momentum_lb,
                "sma_lookback": sma_lb,
                "regime_lookback": regime_lb,
                "fallback_strategy": fallback_col,
                "total_lookback": momentum_lb + sma_lb + regime_lb,

                "train_windows": train_summary["n_windows"],
                "train_improvement_pct": train_summary["improvement_pct"],
                "train_extra_spd_sum_vs_dca": train_summary["extra_spd_sum_vs_dca"],
                "train_win_rate_pct": train_summary["win_rate_pct"],
                "train_strategy_sats": train_summary["strategy_sats"],
                "train_dca_sats": train_summary["dca_sats"],
                "train_fallback_days": train_fallback_days,

                "error": "",
            })

        except Exception as exc:
            lookback_grid_rows.append({
                "status": "error",
                "momentum_lookback": momentum_lb,
                "sma_lookback": sma_lb,
                "regime_lookback": regime_lb,
                "fallback_strategy": fallback_col,
                "total_lookback": momentum_lb + sma_lb + regime_lb,

                "train_windows": np.nan,
                "train_improvement_pct": np.nan,
                "train_extra_spd_sum_vs_dca": np.nan,
                "train_win_rate_pct": np.nan,
                "train_strategy_sats": np.nan,
                "train_dca_sats": np.nan,
                "train_fallback_days": np.nan,

                "error": str(exc)[:300],
            })

lookback_grid_results_df = pd.DataFrame(lookback_grid_rows)

successful_grid_df = lookback_grid_results_df[
    lookback_grid_results_df["status"] == "success"
].copy()

if successful_grid_df.empty:
    print("No successful grid combinations were found. Showing the first 5 error rows:")
    pd.set_option("display.max_colwidth", 300)

    display(
        lookback_grid_results_df[
            [
                "momentum_lookback",
                "sma_lookback",
                "regime_lookback",
                "fallback_strategy",
                "error",
            ]
        ].head(5)
    )

    print("\nMost common error messages:")

    display(
        lookback_grid_results_df["error"]
        .value_counts()
        .head(5)
        .reset_index()
        .rename(columns={"index": "error", "error": "count"})
    )

    raise ValueError(
        "No successful lookback/fallback combinations were found. "
        "See the error column above."
    )


# ============================================================
# Rank combinations using TRAIN metrics only
# ============================================================

ranked_lookback_grid_df = (
    successful_grid_df
    .sort_values(
        [
            "train_improvement_pct",
            "train_win_rate_pct",
            "total_lookback",
        ],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

ranked_lookback_grid_df["rank"] = np.arange(
    1,
    len(ranked_lookback_grid_df) + 1,
)

best_lookback_row = ranked_lookback_grid_df.iloc[0]

best_combo_key = (
    int(best_lookback_row["momentum_lookback"]),
    int(best_lookback_row["sma_lookback"]),
    int(best_lookback_row["regime_lookback"]),
    str(best_lookback_row["fallback_strategy"]),
)

# Overwrite global lookback and fallback settings with selected best values.
MOMENTUM_LOOKBACK = best_combo_key[0]
SMA_LOOKBACK = best_combo_key[1]
REGIME_LOOKBACK = best_combo_key[2]
FALLBACK_STRATEGY = best_combo_key[3]

LOOKBACK_DAYS = sorted({
    MOMENTUM_LOOKBACK,
    SMA_LOOKBACK,
    REGIME_LOOKBACK,
})

CANDIDATE_COLS = get_candidate_cols(SMA_LOOKBACK)
FALLBACK_CANDIDATE_COLS = get_fallback_strategy_cols(SMA_LOOKBACK)


# ============================================================
# Run selected best combo once with TEST included
# ============================================================
# This is the first time test performance is calculated.

selected_result = run_full_regime_strategy_for_lookbacks(
    momentum_lookback=MOMENTUM_LOOKBACK,
    sma_lookback=SMA_LOOKBACK,
    regime_lookback=REGIME_LOOKBACK,
    fallback_strategy=FALLBACK_STRATEGY,
    total_budget_usd=TOTAL_BUDGET_USD,
    include_test=True,
)


# ============================================================
# Pull selected artifacts into final variable names
# ============================================================

btc_data = selected_result["btc_data"]
split_summary_df = selected_result["split_summary_df"]

train_eval_df = selected_result["train_eval_df"]
test_eval_df = selected_result["test_eval_df"]

train_calendar_windows_df = selected_result["train_calendar_windows_df"]
test_calendar_windows_df = selected_result["test_calendar_windows_df"]

train_skipped_years_df = selected_result["train_skipped_years_df"]
test_skipped_years_df = selected_result["test_skipped_years_df"]

train_regime_results_df = selected_result["train_regime_results_df"]
best_mapping_df = selected_result["best_mapping_df"]

train_strategy_df = selected_result["train_strategy_df"]
test_strategy_df = selected_result["test_strategy_df"]

train_window_summary_df = selected_result["train_window_summary_df"]
test_window_summary_df = selected_result["test_window_summary_df"]

train_spd_summary = selected_result["train_spd_summary"]
test_spd_summary = selected_result["test_spd_summary"]


# ============================================================
# Display results
# ============================================================

print("Best lookback + fallback combination selected using TRAIN metrics only:")
print(f"MOMENTUM_LOOKBACK = {MOMENTUM_LOOKBACK}")
print(f"SMA_LOOKBACK      = {SMA_LOOKBACK}")
print(f"REGIME_LOOKBACK   = {REGIME_LOOKBACK}")
print(f"FALLBACK_STRATEGY = {FALLBACK_STRATEGY}")

print("\nFallback strategy options tested for selected SMA lookback:")
print(FALLBACK_CANDIDATE_COLS)

print("\nTop 5 lookback + fallback combinations ranked by TRAIN improvement only:")
display(
    ranked_lookback_grid_df[
        [
            "rank",
            "momentum_lookback",
            "sma_lookback",
            "regime_lookback",
            "fallback_strategy",

            # Train-only selection metrics
            "train_improvement_pct",
            "train_win_rate_pct",
            "train_extra_spd_sum_vs_dca",
            "train_fallback_days",

            "total_lookback",
        ]
    ]
    .head(5)
    .round(6)
)

print("\nSelected train/test split summary:")
display(split_summary_df)

print("\nSelected TRAIN summary:")
display(pd.DataFrame([train_spd_summary]).round(6))

print("\nSelected TEST summary calculated once after train-only selection:")
display(pd.DataFrame([test_spd_summary]).round(6))

print("\nSelected regime sample:")
display(
    btc_data[["date", "price_usd", "combined_regime"]]
    .head()
)

2018: exported 365 rows
2018: exported 365 rows
2018: exported 365 rows
2019: exported 365 rows
2019: exported 365 rows
2019: exported 365 rows
2020: exported 365 rows
2020: exported 365 rows
2020: exported 365 rows
2021: exported 365 rows
2021: exported 365 rows
2021: exported 365 rows
2022: exported 365 rows
2022: exported 365 rows
2022: exported 365 rows
2023: exported 365 rows
2023: exported 365 rows
2023: exported 365 rows
2024: exported 365 rows
2024: exported 365 rows
2024: exported 365 rows
2025: exported 365 rows
2025: exported 365 rows
2025: exported 365 rows
Best lookback + fallback combination selected using TRAIN metrics only:
MOMENTUM_LOOKBACK = 21
SMA_LOOKBACK      = 340
REGIME_LOOKBACK   = 340
FALLBACK_STRATEGY = sma_340d_weight

Fallback strategy options tested for selected SMA lookback:
['sma_340d_weight', 'stacksats_mvrv_weight', 'stacksats_momentum_weight']

Top 5 lookback + fallback combinations ranked by TRAIN improvement only:


,rank,momentum_lookback,sma_lookback,regime_lookback,fallback_strategy,train_improvement_pct,train_win_rate_pct,train_extra_spd_sum_vs_dca,train_fallback_days,total_lookback
0,1,21,340,340,sma_340d_weight,8.829904,83.333333,4460.227898,38,701
1,2,45,340,340,sma_340d_weight,8.619467,83.333333,4353.930397,18,725
2,3,30,340,340,sma_340d_weight,8.613531,83.333333,4350.931504,30,710
3,4,180,340,340,sma_340d_weight,8.607177,83.333333,4347.722229,64,860
4,5,60,340,340,sma_340d_weight,8.600324,83.333333,4344.260509,21,740



Selected train/test split summary:


,split,start_date,end_date,rows_total,rows_eval,windows,calendar_window_start_dates,calendar_window_end_dates,skipped_years,budget_rule
0,train,2018-01-01,2023-12-31,2191,2191,6,"2018-01-01, 2019-01-01, 2020-01-01, 2021-01-01...","2018-12-31, 2019-12-31, 2020-12-31, 2021-12-31...",,"$1,000 per calendar-year training window; Stac..."
1,test,2024-01-01,2025-12-31,731,731,2,"2024-01-01, 2025-01-01","2024-12-31, 2025-12-31",,"$1,000 per calendar-year test window; StackSat..."



Selected TRAIN summary:


,n_windows,wins,losses,ties,win_rate_pct,strategy_sats,dca_sats,extra_sats_vs_dca,strategy_spd_sum,dca_spd_sum,extra_spd_sum_vs_dca,strategy_spd_avg,dca_spd_avg,extra_spd_avg_vs_dca,spd_ratio,improvement_pct
0,6,5,1,0,83.333333,5.497298e+07,5.051275e+07,4.460228e+06,54972.982064,50512.754166,4460.227898,9162.163677,8418.792361,743.371316,1.088299,8.829904



Selected TEST summary calculated once after train-only selection:


,n_windows,wins,losses,ties,win_rate_pct,strategy_sats,dca_sats,extra_sats_vs_dca,strategy_spd_sum,dca_spd_sum,extra_spd_sum_vs_dca,strategy_spd_avg,dca_spd_avg,extra_spd_avg_vs_dca,spd_ratio,improvement_pct
0,2,2,0,0,100.0,2.739160e+06,2.584244e+06,154915.900256,2739.160298,2584.244397,154.9159,1369.580149,1292.122199,77.45795,1.059946,5.99463



Selected regime sample:


,date,price_usd,combined_regime
0,2011-08-16,11.05,BTC Bull | Normal MVRV | Realized Growth Leading
1,2011-08-17,10.88,BTC Bull | Normal MVRV | Realized Growth Leading
2,2011-08-18,10.90,BTC Bull | Normal MVRV | Realized Growth Leading
3,2011-08-19,11.40,BTC Bull | Normal MVRV | Realized Growth Leading
4,2011-08-20,11.49,BTC Bull | Normal MVRV | Realized Growth Leading


In [13]:
# ============================================================
# Cell 10: Build train/test strategy objects
# ============================================================
# Run this full cell after the train-only grid search and selected-combo
# train/test evaluation in Cell 9 have finished.
#
# Important:
# - The regime mapping was learned using TRAIN data only.
# - test_strategy_df is created only after the best train-selected combo is fixed.
# - multi_strategy_regime_train uses TRAIN-period weights from train_strategy_df.
# - multi_strategy_regime uses TEST-period weights from test_strategy_df.
# - Final reporting is handled in Cell 11 using strategy_utils.run_year_by_year().

required_names = [
    "train_strategy_df",
    "test_strategy_df",
    "btc_df",
]

missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Please run the earlier notebook cells first. Missing required object(s): "
        + ", ".join(missing_names)
    )

# Import the StackSats base strategy so we inherit the built-in compute_weights().
# The runner does not allow custom compute_weights() overrides.
try:
    from stacksats.strategy_types import BaseStrategy
except ImportError:
    try:
        from stacksats.strategies.base import BaseStrategy
    except ImportError:
        from stacksats.strategy.base import BaseStrategy

btc_df_pl = btc_df


# ============================================================
# Helper: build lookup table from strategy dataframe
# ============================================================

def build_weight_lookup_from_strategy_df(strategy_df):
    """
    Convert train_strategy_df or test_strategy_df into the lookup format
    expected by MultiStrategyRegimeBasedStrategy.
    """

    weight_lookup_df = (
        strategy_df[
            [
                "date",
                "final_strategy_weight",
                "combined_regime",
                "selected_strategy",
                "used_fallback_strategy",
            ]
        ]
        .copy()
        .assign(date=lambda df: pd.to_datetime(df["date"]).dt.normalize())
        .sort_values("date")
        .drop_duplicates(subset=["date"], keep="last")
        .rename(columns={"final_strategy_weight": "weight"})
        .reset_index(drop=True)
    )

    # Keep weights valid and safe.
    weight_lookup_df["weight"] = (
        weight_lookup_df["weight"]
        .astype(float)
        .clip(lower=0.0)
    )

    return weight_lookup_df


# ============================================================
# Build TRAIN and TEST lookup tables separately
# ============================================================
# Train lookup:
# - Used only for train-period reporting/checking.
#
# Test lookup:
# - Used only for final test comparison after train-only selection.

train_weight_lookup_df = build_weight_lookup_from_strategy_df(train_strategy_df)
test_weight_lookup_df = build_weight_lookup_from_strategy_df(test_strategy_df)

print("Train weight lookup period:")
print("Start:", train_weight_lookup_df["date"].min())
print("End:  ", train_weight_lookup_df["date"].max())
print("Rows: ", len(train_weight_lookup_df))
print("Total weight sum:", train_weight_lookup_df["weight"].sum())

print("\nTest weight lookup period:")
print("Start:", test_weight_lookup_df["date"].min())
print("End:  ", test_weight_lookup_df["date"].max())
print("Rows: ", len(test_weight_lookup_df))
print("Total weight sum:", test_weight_lookup_df["weight"].sum())

print("\nTrain weight sum by year:")
display(
    train_weight_lookup_df
    .assign(year=lambda df: df["date"].dt.year)
    .groupby("year", as_index=False)["weight"]
    .sum()
    .round(6)
)

print("\nTest weight sum by year:")
display(
    test_weight_lookup_df
    .assign(year=lambda df: df["date"].dt.year)
    .groupby("year", as_index=False)["weight"]
    .sum()
    .round(6)
)


class MultiStrategyRegimeBasedStrategy(BaseStrategy):
    """
    StackSats-compatible wrapper for the selected multi-strategy regime approach.

    Important:
    - The regime mapping has already been learned in the earlier cells.
    - This class supplies one weight per date using propose_weight(state).
    - It intentionally does not override compute_weights(), because current
      StackSats versions block custom compute_weights() overrides.
    - It lets strategy_utils.run_year_by_year() export the selected daily
      weights through the standard StackSats runner.export() flow.
    """

    strategy_id = "multi-strategy-regime-based"
    version = "1.0.0"
    description = (
        "Selected multi-strategy regime-based allocation using learned "
        "regime mapping and fallback strategy."
    )

    def __init__(
        self,
        weight_lookup_df: pd.DataFrame,
        source_name: str = "selected_regime_strategy_weights",
    ):
        super().__init__()

        lookup_df = (
            weight_lookup_df[["date", "weight"]]
            .copy()
            .assign(date=lambda df: pd.to_datetime(df["date"]).dt.normalize())
            .sort_values("date")
            .drop_duplicates(subset=["date"], keep="last")
            .reset_index(drop=True)
        )

        self.source_name = source_name

        # Dictionary lookup is simpler and faster inside propose_weight().
        self.weight_by_date = dict(
            zip(
                lookup_df["date"].dt.strftime("%Y-%m-%d"),
                lookup_df["weight"].astype(float),
            )
        )

    def params(self):
        """Keep provenance compact instead of storing the full daily lookup table."""
        return {
            "lookup_days": len(self.weight_by_date),
            "source": self.source_name,
        }

    def _date_from_value(self, value):
        """Convert any date-like value into a YYYY-MM-DD string."""
        if value is None:
            return None

        try:
            return pd.to_datetime(value).normalize().strftime("%Y-%m-%d")
        except Exception:
            return None

    def _extract_state_date(self, state):
        """
        Extract the current date from the StackSats strategy state.

        This is intentionally flexible because StackSats state objects may be
        plain objects, dataclasses, dictionaries, or row-like containers.
        """

        possible_date_fields = [
            "date",
            "current_date",
            "timestamp",
            "time",
            "as_of_date",
            "end_date",
            "start_date",
        ]

        # Dictionary-like state.
        if isinstance(state, dict):
            for key in possible_date_fields:
                date_key = self._date_from_value(state.get(key))
                if date_key is not None:
                    return date_key

            # Nested row/features dictionary fallback.
            for nested_key in ["row", "features", "data", "btc_row"]:
                nested = state.get(nested_key)

                if isinstance(nested, dict):
                    for key in possible_date_fields:
                        date_key = self._date_from_value(nested.get(key))
                        if date_key is not None:
                            return date_key

        # Object/dataclass-like state.
        for attr in possible_date_fields:
            if hasattr(state, attr):
                date_key = self._date_from_value(getattr(state, attr))
                if date_key is not None:
                    return date_key

        # Nested object fallback.
        for nested_attr in ["row", "features", "data", "btc_row"]:
            if hasattr(state, nested_attr):
                nested = getattr(state, nested_attr)

                if isinstance(nested, dict):
                    for key in possible_date_fields:
                        date_key = self._date_from_value(nested.get(key))
                        if date_key is not None:
                            return date_key

                for attr in possible_date_fields:
                    if hasattr(nested, attr):
                        date_key = self._date_from_value(getattr(nested, attr))
                        if date_key is not None:
                            return date_key

        raise ValueError(
            "Could not extract date from StackSats state object. "
            f"State type received: {type(state)}"
        )

    def propose_weight(self, state):
        """
        StackSats calls this once per date.
        Return the selected regime-based allocation weight for that date.
        """

        date_key = self._extract_state_date(state)
        return float(self.weight_by_date.get(date_key, 0.0))

    def to_weights(self, btc_df_input):
        """Manual helper for checking weights outside StackSats."""

        if isinstance(btc_df_input, pl.DataFrame):
            dates = btc_df_input.select(pl.col("date")).to_pandas()["date"]
        else:
            dates = pd.to_datetime(btc_df_input["date"])

        out = pd.DataFrame({"date": pd.to_datetime(dates).dt.normalize()})
        out["date_key"] = out["date"].dt.strftime("%Y-%m-%d")

        out["weight"] = (
            out["date_key"]
            .map(self.weight_by_date)
            .fillna(0.0)
            .astype(float)
        )

        return pl.from_pandas(out[["date", "weight"]])


# ============================================================
# Create train and test strategy objects
# ============================================================

# Used only for train-period reporting/checking.
multi_strategy_regime_train = MultiStrategyRegimeBasedStrategy(
    train_weight_lookup_df,
    source_name="selected_train_regime_strategy_weights",
)

# Used for final test comparison.
multi_strategy_regime = MultiStrategyRegimeBasedStrategy(
    test_weight_lookup_df,
    source_name="selected_test_regime_strategy_weights",
)

print("\nStrategy objects created:")
print("multi_strategy_regime_train")
print("multi_strategy_regime")

Train weight lookup period:
Start: 2018-01-01 00:00:00
End:   2023-12-31 00:00:00
Rows:  2190
Total weight sum: 6.0

Test weight lookup period:
Start: 2024-01-02 00:00:00
End:   2025-12-31 00:00:00
Rows:  730
Total weight sum: 2.0000000000000004

Train weight sum by year:


,year,weight
0,2018,1.0
1,2019,1.0
2,2020,1.0
3,2021,1.0
4,2022,1.0
5,2023,1.0



Test weight sum by year:


,year,weight
0,2024,1.0
1,2025,1.0



Strategy objects created:
multi_strategy_regime_train
multi_strategy_regime


In [16]:
# ============================================================
# Cell 11: Train/test comparison using strategy_utils helper functions
# ============================================================
# This cell uses the reusable helper functions from src.strategy_utils:
# - run_year_by_year()
# - compute_performance_summary()
#
# This keeps the final reporting format consistent with other strategy notebooks.
#
# Important:
# - Train comparison uses multi_strategy_regime_train for 2018–2023.
# - Test comparison uses multi_strategy_regime for 2024–2025.
# - The regime mapping was learned from TRAIN data only.
# - The selected combo was chosen using TRAIN metrics only in Cell 9.

required_names = [
    "multi_strategy_regime_train",
    "multi_strategy_regime",
    "btc_df_pl",
    "runner",
    "src_config",
    "strategy_utils",
    "UniformStrategy",
]

missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Please run the earlier notebook cells first. Missing required object(s): "
        + ", ".join(missing_names)
    )


# ------------------------------------------------------------
# Train comparison: 2018–2023
# ------------------------------------------------------------

strategies_train = {
    "dynamic": multi_strategy_regime_train,
    "baseline": UniformStrategy(),
}

train_years = range(
    src_config.TRAIN_START_YEAR,
    src_config.SPLIT_YEAR,
)  # 2018–2023

merged_train = strategy_utils.run_year_by_year(
    strategies_train,
    btc_df_pl,
    train_years,
    runner,
)

perf_vf = strategy_utils.compute_performance_summary(
    merged_train,
    "dynamic",
    "baseline",
)

print(f"[Train] Multi-strategy SPD : {perf_vf['sats_per_dollar_dynamic']:.2f}")
print(f"[Train] Baseline SPD       : {perf_vf['sats_per_dollar_baseline']:.2f}")
print(
    f"[Train] Multi-strategy is {abs(perf_vf['pct_diff_vs_baseline']):.2f}% "
    f"{perf_vf['performance_label']} than Uniform"
)


# ------------------------------------------------------------
# Test comparison: 2024–2025
# ------------------------------------------------------------

strategies_test = {
    "dynamic": multi_strategy_regime,
    "baseline": UniformStrategy(),
}

test_years = range(
    src_config.SPLIT_YEAR,
    src_config.TEST_END_YEAR + 1,
)  # 2024–2025

merged_test = strategy_utils.run_year_by_year(
    strategies_test,
    btc_df_pl,
    test_years,
    runner,
)

perf_test_vf = strategy_utils.compute_performance_summary(
    merged_test,
    "dynamic",
    "baseline",
)

print(f"\n[Test] Multi-strategy SPD : {perf_test_vf['sats_per_dollar_dynamic']:.2f}")
print(f"[Test] Baseline SPD       : {perf_test_vf['sats_per_dollar_baseline']:.2f}")
print(
    f"[Test] Multi-strategy is {abs(perf_test_vf['pct_diff_vs_baseline']):.2f}% "
    f"{perf_test_vf['performance_label']} than Uniform"
)


# ------------------------------------------------------------
# Display compact summary table
# ------------------------------------------------------------

strategy_utils_summary_df = pd.DataFrame([
    {
        "period": "train",
        "start_year": src_config.TRAIN_START_YEAR,
        "end_year": src_config.SPLIT_YEAR - 1,
        "dynamic_spd": perf_vf["sats_per_dollar_dynamic"],
        "baseline_spd": perf_vf["sats_per_dollar_baseline"],
        "pct_diff_vs_baseline": perf_vf["pct_diff_vs_baseline"],
        "performance_label": perf_vf["performance_label"],
    },
    {
        "period": "test",
        "start_year": src_config.SPLIT_YEAR,
        "end_year": src_config.TEST_END_YEAR,
        "dynamic_spd": perf_test_vf["sats_per_dollar_dynamic"],
        "baseline_spd": perf_test_vf["sats_per_dollar_baseline"],
        "pct_diff_vs_baseline": perf_test_vf["pct_diff_vs_baseline"],
        "performance_label": perf_test_vf["performance_label"],
    },
])

display(strategy_utils_summary_df.round(6))

2018: exported 365 rows
2018: exported 365 rows
2019: exported 365 rows
2019: exported 365 rows
2020: exported 365 rows
2020: exported 365 rows
2021: exported 365 rows
2021: exported 365 rows
2022: exported 365 rows
2022: exported 365 rows
2023: exported 365 rows
2023: exported 365 rows
[Train] Multi-strategy SPD : 9161.44
[Train] Baseline SPD       : 8418.79
[Train] Multi-strategy is 8.82% better than Uniform
2024: exported 365 rows
2024: exported 365 rows
2025: exported 365 rows
2025: exported 365 rows

[Test] Multi-strategy SPD : 1369.57
[Test] Baseline SPD       : 1292.12
[Test] Multi-strategy is 5.99% better than Uniform


,period,start_year,end_year,dynamic_spd,baseline_spd,pct_diff_vs_baseline,performance_label
0,train,2018,2023,9161.435122,8418.792361,8.82125,better
1,test,2024,2025,1369.573298,1292.122199,5.99410,better


In [25]:
# ============================================================
# Cell 12: Compare selected strategy against Uniform DCA on test period
# ============================================================
# This cell compares the final selected multi-strategy regime approach
# against the UniformStrategy baseline using StackSats runner.compare().


from stacksats import UniformStrategy, ComparisonConfig

uniform_strategy = UniformStrategy()

result_test = runner.compare(
    strategies=[
        multi_strategy_regime,
        uniform_strategy,
    ],
    config=ComparisonConfig(
        start_date=TEST_START,
        end_date=TEST_END,
        baseline="uniform",
        strict=False,
        output_dir="output",
    ),
    btc_df=btc_df_pl,
)


In [26]:
# ============================================================
# Cell 13
# ============================================================
result_test_df = result_test.to_dataframe()
result_test_df

selector,strategy_id,strategy_version,intent_mode,tier,promotion_stage,validation_passed,judgment_label,win_rate,score,exp_decay_percentile,multiple_vs_uniform,score_delta_vs_baseline,exp_decay_delta_vs_baseline,is_baseline
str,str,str,str,str,str,bool,str,f64,f64,f64,f64,f64,f64,bool
"""multi-strategy-regime-based""","""multi-strategy-regime-based""","""1.0.0""","""propose""",null,null,true,"""validation-passed""",63.760218,51.239992,38.719766,1.018343,32.228831,0.697443,false
"""uniform""","""uniform""","""1.0.0""","""propose""","""stable""","""promoted""",false,"""validation-failed""",0.0,19.011162,38.022323,1.0,0.0,0.0,true


In [27]:
# ============================================================
# Cell 14: Full-period plot using merged train/test helper outputs
# ============================================================
# This cell uses the outputs from strategy_utils.run_year_by_year():
# - merged_train
# - merged_test
#
# The plot uses src.plots.plot_strategy_full_period(), so we do not need
# to manually rebuild strategy/baseline plotting columns.

required_names = [
    "merged_train",
    "merged_test",
    "plots",
]

missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Please run Cell 11 first. Missing required object(s): "
        + ", ".join(missing_names)
    )

# Convert Polars output to pandas if needed.
merged_train_pd = (
    merged_train.to_pandas()
    if hasattr(merged_train, "to_pandas")
    else merged_train.copy()
)

merged_test_pd = (
    merged_test.to_pandas()
    if hasattr(merged_test, "to_pandas")
    else merged_test.copy()
)

combined_plot_df = (
    pd.concat(
        [
            merged_train_pd,
            merged_test_pd,
        ],
        ignore_index=True,
    )
    .sort_values("date")
    .reset_index(drop=True)
)

combined_plot_df["date"] = pd.to_datetime(combined_plot_df["date"])

cols = plots.StrategyColumns(
    weight="dynamic_weight",
    spd="sats_per_dollar_dynamic",
    sats_accum="sats_accum_dynamic",
)

# Full-period plot: train + test.
full_plot = plots.plot_strategy_full_period(
    combined_plot_df,
    cols,
    "Multi-strategy Regime-Based Approach",
    date_range=("2018-01-01", str(combined_plot_df["date"].max().date())),
    test_start_date="2024-01-01",
)

plt.show()